# FEVERScope: Pinecone Semantic Vector Indexing (Google Colab)

This standalone Colab notebook runs on **GPU** to:
1. Encode FEVER evidence sentences with `sentence-transformers` (`all-MiniLM-L6-v2`, 384 dimensions).
2. Create a serverless **Pinecone** index.
3. Upsert dense sentence vectors with metadata (`doc_id`, `page_title`, `sent_id`, `text`).

> **GPU Recommended**: Go to `Runtime` -> `Change runtime type` -> select `T4 GPU`.

In [ ]:
# Step 1: Install required libraries
!pip install -q sentence-transformers pinecone-client pandas tqdm pyarrow

In [ ]:
# Step 2: Configure Pinecone Credentials
import os

PINECONE_API_KEY = "YOUR_PINECONE_API_KEY"       # <-- Replace with your Pinecone API key
PINECONE_INDEX_NAME = "fever-evidence-index"      # <-- Your chosen index name
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"         # 384 dimensions
VECTOR_DIMENSION = 384
BATCH_SIZE = 100                                  # Upsert batch size

In [ ]:
# Step 3: Initialize Pinecone client & Create Index
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

# Check existing indexes
existing_indexes = [idx["name"] for idx in pc.list_indexes()]
print(f"Existing indexes: {existing_indexes}")

if PINECONE_INDEX_NAME not in existing_indexes:
    print(f"Creating index: {PINECONE_INDEX_NAME} (dimension={VECTOR_DIMENSION}, metric=cosine)...")
    pc.create_index(
        name=PINECONE_INDEX_NAME,
        dimension=VECTOR_DIMENSION,
        metric="cosine",
        spec=ServerlessSpec(cloud="aws", region="us-east-1")
    )
    print("Index created successfully!")
else:
    print(f"Index {PINECONE_INDEX_NAME} already exists. Ready to connect.")

index = pc.Index(PINECONE_INDEX_NAME)
print(index.describe_index_stats())

In [ ]:
# Step 4: Load Sentence Transformer on GPU
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

model = SentenceTransformer(EMBEDDING_MODEL_NAME, device=device)
print(f"Loaded {EMBEDDING_MODEL_NAME} on {device}.")

In [ ]:
# Step 5: Load or Upload Corpus Data
# You can upload your 'mini_corpus.parquet' or 'corpus.parquet' via the file upload widget below,
# or mount Google Drive.
from google.colab import files
import pandas as pd

print("Upload your corpus file (e.g. mini_corpus.parquet or corpus.parquet):")
uploaded = files.upload()
corpus_filename = list(uploaded.keys())[0]
df = pd.read_parquet(corpus_filename)

print(f"Loaded {len(df):,} sentences from {corpus_filename}.")
df.head(3)

In [ ]:
# Step 6: Encode and Upsert to Pinecone in Batches
from tqdm.auto import tqdm

print(f"Upserting {len(df):,} records into Pinecone...")

for start_idx in tqdm(range(0, len(df), BATCH_SIZE)):
    end_idx = min(start_idx + BATCH_SIZE, len(df))
    batch_df = df.iloc[start_idx:end_idx]
    
    # Prefer title-augmented text if available
    texts = (
        batch_df["text_with_title"].tolist()
        if "text_with_title" in batch_df.columns
        else batch_df["text"].tolist()
    )
    
    # Generate embeddings
    embeddings = model.encode(texts, batch_size=BATCH_SIZE, normalize_embeddings=True, show_progress_bar=False)
    
    # Format Pinecone vectors
    vectors_to_upsert = []
    for i, (_, row) in enumerate(batch_df.iterrows()):
        doc_id = str(row["doc_id"])
        vector_values = embeddings[i].tolist()
        metadata = {
            "doc_id": doc_id,
            "page_title": str(row["page_title"]),
            "sent_id": int(row["sent_id"]),
            "text": str(row["text"][:1000]),  # Pinecone metadata size guard
        }
        vectors_to_upsert.append({"id": doc_id, "values": vector_values, "metadata": metadata})
        
    index.upsert(vectors=vectors_to_upsert)

print("Upsert completed successfully!")
print(index.describe_index_stats())

In [ ]:
# Step 7: Sanity Test Query
test_claim = "Roman Empire was one of the largest empires in history."
query_embedding = model.encode(test_claim, normalize_embeddings=True).tolist()

results = index.query(
    vector=query_embedding,
    top_k=5,
    include_metadata=True
)

print(f"\nSearch Results for: '{test_claim}'")
for match in results["matches"]:
    score = match["score"]
    doc_id = match["id"]
    text = match["metadata"].get("text", "")
    print(f"- [{score:.4f}] {doc_id} -> {text[:120]}...")

## Step 8: Next Steps
Copy your `PINECONE_API_KEY` and `PINECONE_INDEX_NAME` back to your FEVERScope `.env` file:
```bash
PINECONE_API_KEY=your_key_here
PINECONE_INDEX_NAME=fever-evidence-index
```
Your FEVERScope FastAPI backend will automatically use this index for Dense Semantic retrieval!